# HealthBridge — 03: SHAP Explainability Analysis


In [ ]:
import sys
sys.path.insert(0, '..')
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from pathlib import Path
from explain import load_artifacts, build_explainer, top_shap_features, waterfall_to_base64, FEATURE_COLS
import base64
ROOT = Path('..')


## 1. Load model artifacts

In [ ]:
pipe, le = load_artifacts()
print(f'Model: {type(pipe.named_steps["clf"]).__name__}')
print(f'Disease classes ({len(le.classes_)}):', le.classes_.tolist())


## 2. Global feature importance (SHAP summary)

In [ ]:
img = mpimg.imread(ROOT / 'reports/shap/summary_beeswarm.png')
plt.figure(figsize=(10, 6))
plt.imshow(img); plt.axis('off')
plt.title('Global Feature Importance — Mean |SHAP|', fontsize=13)
plt.tight_layout(); plt.show()


## 3. Per-sample waterfall plots

In [ ]:
for i in range(3):
    p = ROOT / f'reports/shap/waterfall_sample_{i}.png'
    if p.exists():
        img = mpimg.imread(p)
        plt.figure(figsize=(10, 4))
        plt.imshow(img); plt.axis('off')
        plt.title(f'Waterfall Plot — Sample {i}', fontsize=12)
        plt.tight_layout(); plt.show()


## 4. Interactive: predict a custom patient + see SHAP

In [ ]:
import io, base64
from IPython.display import Image, display

custom_patient = pd.DataFrame([{
    'Age': 65, 'Gender': 1, 'Fever': 0, 'Cough': 0, 'Fatigue': 0,
    'Breathlessness': 0, 'Chest_Pain': 0, 'Nausea': 0, 'Headache': 0,
    'BMI': 28.0, 'BP_Systolic': 168, 'BP_Diastolic': 102,
    'SpO2': 97, 'Glucose_mg_dL': 95, 'HbA1c': 5.3
}])

pred_enc = pipe.predict(custom_patient)[0]
pred_class = le.inverse_transform([pred_enc])[0]
proba = pipe.predict_proba(custom_patient)[0]
conf = proba[pred_enc]
print(f'Prediction: {pred_class}  ({conf*100:.1f}% confidence)')

# SHAP waterfall
b64 = waterfall_to_base64(pipe, le, custom_patient)
display(Image(data=base64.b64decode(b64)))
